# Genomic Needle-in-a-Haystack (gNIAH) Protocol

Insert known regulatory motifs at varying distances in neutral backgrounds
and measure model sensitivity. This directly probes how far a model can
"see" a specific regulatory signal.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from ecl.models.base import SyntheticModel
from ecl.gniah import gniah_sensitivity, MOTIFS

In [ ]:
print("Available motifs:")
for name, seq in MOTIFS.items():
    print(f"  {name}: {seq}")

In [ ]:
# Two models with different context ranges
models = {
    "Short-range model": SyntheticModel(seq_length=500, embed_dim=32, decay_length=30),
    "Long-range model": SyntheticModel(seq_length=500, embed_dim=32, decay_length=150),
}
rng = np.random.default_rng(42)
distances = np.array([5, 10, 20, 50, 80, 120, 180])
motif_names = ["CTCF", "GATA", "SP1"]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {"Short-range model": "#e41a1c", "Long-range model": "#377eb8"}

for mi, motif in enumerate(motif_names):
    ax = axes[mi]
    for name, model in models.items():
        sens = gniah_sensitivity(
            model_fn=model, motif_name=motif, distances=distances,
            seq_length=500, n_samples=10, rng=rng, show_progress=False,
        )
        # Normalize to max
        if sens.max() > 0:
            sens = sens / sens.max()
        ax.plot(distances, sens, 'o-', color=colors[name], label=name, linewidth=1.5)

    ax.set_xlabel('Distance from center (bp)')
    ax.set_ylabel('Normalized gNIAH sensitivity')
    ax.set_title(f'Motif: {motif}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()